In [1]:
!pip install -q transformers datasets accelerate evaluate torchaudio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.7 MB/s eta 0:00:00


In [2]:
import os, json, pickle
import numpy as np
import torch
import torchaudio
from pathlib import Path
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
from transformers import ASTForAudioClassification, ASTFeatureExtractor, TrainingArguments, Trainer
import evaluate

print(f'GPU: {torch.cuda.get_device_name(0)}')

GPU: Tesla T4


In [3]:
# 데이터 로드
PREVIEW_DIR = '/kaggle/input/datasets/zoebae/soundtag-genre-previews/genre_data/previews'

class GenreDataset(torch.utils.data.Dataset):
    def __init__(self, file_paths, labels, feature_extractor, target_sr=16000):
        self.file_paths = file_paths
        self.labels = labels
        self.fe = feature_extractor
        self.sr = target_sr

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        try:
            waveform, sr = torchaudio.load(self.file_paths[idx])
            if waveform.shape[0] > 1:
                waveform = waveform.mean(dim=0, keepdim=True)
            waveform = waveform.squeeze(0)
            if sr != self.sr:
                waveform = torchaudio.transforms.Resample(sr, self.sr)(waveform)
            inputs = self.fe(waveform.numpy(), sampling_rate=self.sr, return_tensors='pt')
            return {'input_values': inputs['input_values'].squeeze(0), 'labels': self.labels[idx]}
        except:
            return {'input_values': torch.zeros(1024, 128), 'labels': self.labels[idx]}

file_paths, genre_labels = [], []
for genre_dir in sorted(Path(PREVIEW_DIR).iterdir()):
    if genre_dir.is_dir():
        genre_name = genre_dir.name.replace('_', ' ').replace('RandB', 'R&B')
        for mp3 in genre_dir.glob('*.mp3'):
            file_paths.append(str(mp3))
            genre_labels.append(genre_name)

le = LabelEncoder()
encoded_labels = le.fit_transform(genre_labels)
train_paths, test_paths, train_labels, test_labels = train_test_split(
    file_paths, encoded_labels, test_size=0.2, random_state=42, stratify=encoded_labels)

fe = ASTFeatureExtractor.from_pretrained('MIT/ast-finetuned-audioset-10-10-0.4593')
train_dataset = GenreDataset(train_paths, train_labels, fe)
test_dataset = GenreDataset(test_paths, test_labels, fe)

print(f'Train: {len(train_paths)}, Test: {len(test_paths)}, Classes: {len(le.classes_)}')

preprocessor_config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

Train: 2080, Test: 520, Classes: 26


In [4]:
# 학습
model = ASTForAudioClassification.from_pretrained(
    'MIT/ast-finetuned-audioset-10-10-0.4593',
    num_labels=len(le.classes_), ignore_mismatched_sizes=True)

accuracy_metric = evaluate.load('accuracy')
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=preds, references=labels)
    top3 = sum(labels[i] in np.argsort(logits[i])[-3:] for i in range(len(labels))) / len(labels)
    acc['top3_accuracy'] = top3
    return acc

trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir='/kaggle/working/ast_model',
        eval_strategy='epoch', save_strategy='epoch',
        learning_rate=1e-5, per_device_train_batch_size=8,
        num_train_epochs=10, warmup_ratio=0.1, weight_decay=0.01,
        load_best_model_at_end=True, metric_for_best_model='accuracy',
        save_total_limit=2, fp16=True, report_to='none'),
    train_dataset=train_dataset, eval_dataset=test_dataset,
    compute_metrics=compute_metrics)

trainer.train()

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

ASTForAudioClassification LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                     | Status   |                                                                                        
------------------------+----------+----------------------------------------------------------------------------------------
classifier.dense.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([527, 768]) vs model:torch.Size([26, 768])
classifier.dense.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([527]) vs model:torch.Size([26])          

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1300, training_loss=1.1716876220703125, metrics={'train_runtime': 5218.3778, 'train_samples_per_second': 3.986, 'train_steps_per_second': 0.249, 'total_flos': 1.4101849690865664e+18, 'train_loss': 1.1716876220703125, 'epoch': 10.0})

In [5]:
# 평가 + 모델 저장 + K-pop 분석 한번에
results = trainer.evaluate()
print(f"\nAccuracy: {results['eval_accuracy']:.1%}, Top-3: {results['eval_top3_accuracy']:.1%}")

preds = trainer.predict(test_dataset)
print(classification_report(preds.label_ids, np.argmax(preds.predictions, axis=-1),
                            target_names=le.classes_, zero_division=0))

# K-pop 분석
KPOP_DIR = '/kaggle/input/datasets/zoebae/soundtag-kpop-previews/deezer_data/previews'
kpop_files = sorted(Path(KPOP_DIR).glob('*.mp3'))
print(f'\nK-pop files: {len(kpop_files)}')

model.eval()
kpop_results = []
for i, mp3 in enumerate(kpop_files):
    try:
        waveform, sr = torchaudio.load(str(mp3))
        if waveform.shape[0] > 1: waveform = waveform.mean(dim=0, keepdim=True)
        waveform = waveform.squeeze(0)
        if sr != 16000: waveform = torchaudio.transforms.Resample(sr, 16000)(waveform)
        inputs = fe(waveform.numpy(), sampling_rate=16000, return_tensors='pt')
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
        with torch.no_grad():
            probs = torch.softmax(model(**inputs).logits, dim=-1).cpu().numpy()[0]
        top5 = [(str(le.classes_[j]), float(probs[j])) for j in np.argsort(probs)[-5:][::-1]]
        kpop_results.append({'filename': mp3.name, 'genres': top5})
        if i < 20:
            print(f'  {mp3.name:45s} → {", ".join(f"{g}({s:.0%})" for g,s in top5[:3])}')
    except Exception as e:
        if i < 3: print(f'  Error: {e}')
    if (i+1) % 200 == 0: print(f'  [{i+1}/{len(kpop_files)}]')

top1 = Counter(r['genres'][0][0] for r in kpop_results)
print(f'\n📊 K-pop 장르 분포:')
for g, c in top1.most_common():
    print(f'  {g:25s} {c:4d} ({c/len(kpop_results)*100:5.1f}%)')

with open('/kaggle/working/ast_kpop_analysis.json', 'w') as f:
    json.dump(kpop_results, f, ensure_ascii=False, indent=2)
print('💾 Saved!')


Accuracy: 42.1%, Top-3: 58.1%
                  precision    recall  f1-score   support

       Afrobeats       0.43      0.50      0.47        20
          Ballad       0.24      0.45      0.31        20
     Bedroom Pop       0.27      0.20      0.23        20
        City Pop       0.29      0.35      0.32        20
Contemporary R&B       0.65      0.65      0.65        20
       Dance Pop       0.42      0.25      0.31        20
           Disco       0.08      0.05      0.06        20
           Drill       0.44      0.35      0.39        20
             EDM       0.14      0.05      0.07        20
      Electropop       0.28      0.35      0.31        20
        Funk Pop       0.50      0.20      0.29        20
     Future Bass       0.20      0.10      0.13        20
         Hip Hop       0.41      0.60      0.49        20
        Hyperpop       0.30      0.45      0.36        20
     Jersey Club       0.81      0.85      0.83        20
       Latin Pop       0.62      0.80   